In [ ]:
# create model class
import torch
import torch.nn as nn

class Model(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features, 3),
        nn.ReLU(),
        nn.Linear(3, 1),
        nn.Sigmoid()
    )

  def forward(self, features):

    out = self.network(features)

    return out

### **Why use `nn.Module`?**

`torch.nn.Module` is the fundamental building block for all neural networks in PyTorch. It provides a structured way to define your network, manage its parameters, and handle various operational aspects. Here are the key reasons why it's essential:

1.  **State Management**: It automatically tracks all learnable parameters (weights and biases) and sub-modules within your network.
2.  **Parameter Registration**: When you assign an `nn.Parameter` or another `nn.Module` as an attribute, `nn.Module` registers them, making them discoverable by optimizers.
3.  **Forward Pass Abstraction**: It provides a standard `forward()` method where you define the computational flow of your network.
4.  **Device Management**: Simplifies moving the model to different devices (CPU/GPU) using methods like `.to(device)`.
5.  **Training/Evaluation Modes**: Supports `.train()` and `.eval()` methods to control layer behavior (e.g., Dropout, BatchNorm) during different phases.
6.  **Saving and Loading**: Facilitates easy saving and loading of model states using `torch.save()` and `torch.load()`.

In [1]:
import torch
import torch.nn as nn

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.10.0+cpu


### **How to use `nn.Module`?**

To create a neural network, you inherit from `nn.Module` and implement at least two core methods:

1.  **`__init__(self, ...)`**: The constructor, where you define all layers and sub-modules. Always call `super().__init__()`.
2.  **`forward(self, input)`**: Defines the forward pass, describing how input data is processed through the layers to produce an output.

In [2]:
# Define a simple neural network by inheriting from nn.Module
class SimpleModel(nn.Module):

  def __init__(self, input_features, hidden_size, output_size):
    super().__init__() # Initialize the base nn.Module class

    # Define the layers of the network
    self.layer1 = nn.Linear(input_features, hidden_size)
    self.relu = nn.ReLU()
    self.layer2 = nn.Linear(hidden_size, output_size)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    # Define the forward pass logic
    x = self.layer1(x)
    x = self.relu(x)
    x = self.layer2(x)
    x = self.sigmoid(x)
    return x

print("SimpleModel class defined.")

SimpleModel class defined.


#### **Explanation of the `SimpleModel`:**

*   **`class SimpleModel(nn.Module):`**: Declares `SimpleModel` as a PyTorch module.
*   **`super().__init__()`**: Crucial call to initialize the parent `nn.Module`.
*   **`self.layer1 = nn.Linear(...)`**, **`self.relu = nn.ReLU()`**, etc.: We define individual layers as attributes. When these layers are `nn.Module` instances themselves, `SimpleModel` automatically registers their parameters.
*   **`def forward(self, x):`**: This method dictates how data `x` flows through `layer1`, `relu`, `layer2`, and `sigmoid` in sequence. When you call an instance of `SimpleModel`, this `forward` method is executed.

In [3]:
# Example usage: Instantiate the model and perform a forward pass
input_features = 10
hidden_size = 5
output_size = 1

# Create an instance of the model
model = SimpleModel(input_features, hidden_size, output_size)

# Create a dummy input tensor
dummy_input = torch.randn(1, input_features) # Batch size 1, 10 features

# Perform a forward pass
output = model(dummy_input)

print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Output values: {output}")

Input shape: torch.Size([1, 10])
Output shape: torch.Size([1, 1])
Output values: tensor([[0.6247]], grad_fn=<SigmoidBackward0>)


#### **Device Management (`.to(device)`)**

`nn.Module` makes it easy to move your entire model (all its parameters and buffers) to a specific device, like a GPU for faster computation, or back to the CPU.

In [4]:
# Check if CUDA (GPU) is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move the model to the chosen device
model.to(device)
print(f"Model moved to {next(model.parameters()).device}")

# Move input tensor to the same device
dummy_input_on_device = dummy_input.to(device)

# Perform forward pass on the device
output_on_device = model(dummy_input_on_device)
print(f"Output device: {output_on_device.device}")

Using device: cpu
Model moved to cpu
Output device: cpu


#### **Training and Evaluation Modes (`.train()` and `.eval()`)**

Certain layers, like `nn.Dropout` and `nn.BatchNorm`, behave differently during training and evaluation. `nn.Module` provides methods to easily switch the mode for all applicable layers in your model.

In [5]:
class ModelWithDropout(nn.Module):
  def __init__(self, input_features):
    super().__init__()
    self.linear1 = nn.Linear(input_features, 10)
    self.dropout = nn.Dropout(0.5) # 50% dropout
    self.linear2 = nn.Linear(10, 1)

  def forward(self, x):
    x = self.linear1(x)
    x = self.dropout(x) # Dropout layer
    x = self.linear2(x)
    return x

dropout_model = ModelWithDropout(input_features)

print("\nInitial state (default is training):")
print(f"Dropout in training mode: {dropout_model.dropout.training}")

# Set the model to evaluation mode
dropout_model.eval()
print("\nAfter calling .eval():")
print(f"Dropout in evaluation mode: {dropout_model.dropout.training}")

# Set the model back to training mode
dropout_model.train()
print("\nAfter calling .train():")
print(f"Dropout in training mode: {dropout_model.dropout.training}")


Initial state (default is training):
Dropout in training mode: True

After calling .eval():
Dropout in evaluation mode: False

After calling .train():
Dropout in training mode: True


### **Accessing Parameters and Model Summary**

`nn.Module` facilitates inspection of your network.


In [6]:
# Accessing parameters directly
print("\nModel parameters:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

# Using torchinfo for a detailed summary (requires installing torchinfo)
!pip install -qqq torchinfo
from torchinfo import summary

print("\nModel summary:")
summary(model, input_size=(1, input_features)) # input_size is (batch_size, num_features)


Model parameters:
  layer1.weight: torch.Size([5, 10])
  layer1.bias: torch.Size([5])
  layer2.weight: torch.Size([1, 5])
  layer2.bias: torch.Size([1])

Model summary:


Layer (type:depth-idx)                   Output Shape              Param #
SimpleModel                              [1, 1]                    --
├─Linear: 1-1                            [1, 5]                    55
├─ReLU: 1-2                              [1, 5]                    --
├─Linear: 1-3                            [1, 1]                    6
├─Sigmoid: 1-4                           [1, 1]                    --
Total params: 61
Trainable params: 61
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

This comprehensive structure provided by `nn.Module` makes PyTorch a powerful and flexible framework for deep learning, allowing you to build, train, and deploy complex models efficiently.

In [ ]:
# create dataset
features = torch.rand(10,5)

# create model
model = Model(features.shape[1])

# call model for forward pass
# model.forward(features)
model(features)

tensor([[0.6188],
        [0.6139],
        [0.6124],
        [0.6172],
        [0.6167],
        [0.6180],
        [0.6211],
        [0.6171],
        [0.6156],
        [0.6220]], grad_fn=<SigmoidBackward0>)

In [ ]:
# show model weights
model.linear2.weight

Parameter containing:
tensor([[0.5021, 0.5738, 0.4083]], requires_grad=True)

In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary

summary(model, input_size=(10, 5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00